# Aula 9 — Sentimentos, NER e Extração de Relações (Lance!)

**Disciplina:** Processamento de Linguagem Natural — FURB · 2026/2  
**Base:** notícias de futebol brasileiro coletadas pelo projeto [`fgiovanella/pln`](https://github.com/fgiovanella/pln).

Este notebook atende **somente às três técnicas da questão 5 do Trabalho da Unidade 3**:

1. **Análise de sentimentos:** identificar tom positivo, neutro ou negativo em uma amostra de notícias.
2. **Reconhecimento de Entidades Nomeadas (NER):** localizar nomes de pessoas, organizações e localidades.
3. **Extração de relações:** analisar dependências sintáticas para extrair triplas candidato (*sujeito — relação → objeto*) e apresentar um grafo.

A abordagem de NER e relações adapta o notebook fornecido na **Aula 9 — Parsing, NER e extração de relações**; a etapa de sentimento é um complemento inspirado na questão 5 do PDF.

**Importante:** os resultados são estimativas de modelos ou regras, **não fatos verificados**. Execute tudo com sua base real antes de escrever a conclusão; nenhuma métrica está preenchida artificialmente. Se usar amostragem, informe seu tamanho e o limite de texto analisado.

## 0. Preparação do ambiente

Abra este notebook **na raiz do repositório `pln`** (ao lado de `pipeline.py`) ou na pasta `Trabalho_Unidade_3` dentro dele. Também funciona no Google Colab **se a base for disponibilizada**.

Instale as bibliotecas. O download do modelo spaCy e do classificador de sentimentos exige internet na primeira execução. Em Colab, pode ser necessário reiniciar o ambiente após instalar.

Se **não houver dados em `outputs/`**, execute na raiz do projeto `python pipeline.py --limite 50` em um terminal, ou transfira o CSV/JSONL gerado no seu computador para uma pasta `outputs`. O notebook **não gera notícias fictícias**.

In [ ]:
%pip -q install pandas matplotlib networkx spacy transformers torch


In [ ]:
!python -m spacy download pt_core_news_sm -q


In [ ]:
import spacy
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display
from spacy import displacy
from pathlib import Path
from collections import Counter
import textwrap

nlp = spacy.load('pt_core_news_sm')
print('spaCy:',spacy.__version__,'| modelo:',nlp.meta.get('version'))
PASTA_TRABALHO = Path('Trabalho_Unidade_3') if Path('Trabalho_Unidade_3').is_dir() else Path('.')
SAIDA = PASTA_TRABALHO / 'resultados'
SAIDA.mkdir(parents=True, exist_ok=True)

## Carregamento da base original do projeto

Usamos os textos completos **antes** de remover stopwords/pontuação; caso contrário, o parser perde sujeito, verbo e dependências importantes. Limitamos a amostra para uma execução exploratória, com tamanho e semente documentados.

In [ ]:
from pathlib import Path
import json
import pandas as pd

CANDIDATOS = [
    "outputs/noticias_lance_processadas.csv",
    "outputs/noticias_lance_processadas.jsonl",
    "outputs/noticias_lance_processadas.json",
    "outputs/noticias_lance.jsonl",
    "outputs/noticias_lance.json",
]


def encontrar_base():
    """Procura os arquivos gerados pelo pipeline original, sem baixar dados fictícios."""
    raizes = [Path.cwd(), *list(Path.cwd().parents)[:3], Path('/content/pln')]
    caminhos = [raiz / item for raiz in raizes for item in CANDIDATOS]
    for caminho in caminhos:
        if caminho.is_file():
            return caminho
    raise FileNotFoundError(
        "Base do Lance! não encontrada. Execute 'python pipeline.py --limite 150' "
        "na raiz do repositório, depois execute o notebook novamente. "
        "Caminhos tentados: " + ", ".join(str(x) for x in caminhos[:8])
    )


def ler_base(caminho):
    if caminho.suffix == ".csv":
        df = pd.read_csv(caminho, encoding='utf-8-sig')
    elif caminho.suffix == ".jsonl":
        df = pd.read_json(caminho, lines=True)
    else:
        with caminho.open(encoding='utf-8') as f:
            dados = json.load(f)
        if isinstance(dados, dict):
            lista = next((dados[chave] for chave in (
                'noticias', 'artigos', 'materias', 'dados', 'registros', 'items'
            ) if isinstance(dados.get(chave), list)), None)
            if lista is None:
                raise ValueError(f"Não encontrei lista de artigos em {caminho}; chaves: {list(dados)}")
            dados = lista
        df = pd.DataFrame(dados)
    if df.empty:
        raise ValueError(f"Arquivo sem notícias: {caminho}")
    return df


def padronizar_base(df):
    # NER e parsing exigem texto legível, não stemming ou BoW pré-processado.
    textos = ('texto_original', 'texto', 'conteudo', 'conteúdo', 'corpo',
              'texto_completo', 'corpo_texto', 'body', 'content',
              'texto_limpo', 'texto_processado')
    cols = {str(c).lower(): c for c in df.columns}
    coluna = next((cols[x] for x in textos if x in cols), None)
    if coluna is None:
        raise ValueError(f"Coluna de texto não encontrada. Disponíveis: {list(df.columns)}")
    print(f"Coluna de texto selecionada: {coluna!r}. Confirme que é texto original para NER.")
    out = df.copy()
    out['_texto_unidade3'] = out[coluna].fillna('').astype(str).str.strip()
    out = out[out['_texto_unidade3'].str.len() >= 70].copy().reset_index(drop=True)
    if out.empty:
        raise ValueError("Não há textos com pelo menos 70 caracteres após a limpeza.")
    out['_id_unidade3'] = out.index.astype(int)
    for destino, opcoes in {
        '_titulo_unidade3': ('titulo','título','title'),
        '_url_unidade3': ('url','link'),
    }.items():
        origem = next((cols[o] for o in opcoes if o in cols), None)
        out[destino] = out[origem].fillna('').astype(str) if origem else ''
    return out

caminho_base = encontrar_base()
base = padronizar_base(ler_base(caminho_base))
print(f'Base: {caminho_base} | notícias válidas: {len(base)}')
print('Colunas disponíveis:', list(base.columns))
display(base[['_id_unidade3','_titulo_unidade3','_url_unidade3']].head())

In [ ]:
SEMENTE = 42
N_NOTICIAS = min(25, len(base))
LIMITE_CARACTERES = 2500
amostra = base.sample(n=N_NOTICIAS, random_state=SEMENTE).reset_index(drop=True)
print(f'Amostra NER: {N_NOTICIAS} / {len(base)} notícias; semente {SEMENTE}; até {LIMITE_CARACTERES} caracteres por notícia.')

## 1. Análise de sentimentos

O modelo pré-treinado **`cardiffnlp/twitter-xlm-roberta-base-sentiment`** prevê a polaridade **negativa, neutra ou positiva** em texto multilíngue. É um modelo treinado para postagens em redes sociais, **não foi treinado especificamente com notícias do Lance!**. Por isso, o resultado deve ser visto como exploratório, e não como identificação confiável do sentimento de torcedores, do autor ou de quem venceu o jogo.

Para reduzir tempo e memória, classificamos **título + começo do texto**, com truncamento em até 256 tokens. Se o texto for mais longo, o modelo não considera tudo. O tamanho da amostra e o corte de tokens ficam explicitados na saída.

In [ ]:
from transformers import pipeline

MODELO_SENTIMENTOS = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
N_SENTIMENTOS = min(15, len(amostra))
MAX_TOKENS_SENTIMENTOS = 256

# No primeiro uso o modelo será baixado do Hugging Face (precisa de internet).
classificador_sentimentos = pipeline(
    task='text-classification',
    model=MODELO_SENTIMENTOS,
    device=-1,  # CPU, para funcionar em computadores sem GPU dedicada
)

def montar_texto_sentimento(linha):
    titulo = str(linha['_titulo_unidade3']).strip()
    texto = str(linha['_texto_unidade3']).strip()
    return f'{titulo}. {texto}' if titulo else texto

sentimentos_amostra = amostra.head(N_SENTIMENTOS).copy()
entradas = [montar_texto_sentimento(linha) for _, linha in sentimentos_amostra.iterrows()]
previsoes = classificador_sentimentos(
    entradas, truncation=True, max_length=MAX_TOKENS_SENTIMENTOS,
    batch_size=4,
)

# Rótulos de alguns checkpoints aparecem como LABEL_0, LABEL_1 e LABEL_2.
rotulos = {'LABEL_0': 'negativo', 'LABEL_1': 'neutro', 'LABEL_2': 'positivo',
           'negative': 'negativo', 'neutral': 'neutro', 'positive': 'positivo'}

sentimentos_df = pd.DataFrame({
    'id_amostra': sentimentos_amostra.index,
    'titulo': sentimentos_amostra['_titulo_unidade3'].tolist(),
    'url': sentimentos_amostra['_url_unidade3'].tolist(),
    'sentimento': [rotulos.get(str(p['label']), str(p['label'])) for p in previsoes],
    'confianca_modelo': [float(p['score']) for p in previsoes],
    'rotulo_original': [p['label'] for p in previsoes],
    'revisao_manual': [''] * len(previsoes),
})
sentimentos_df.to_csv(SAIDA / 'sentimentos_lance.csv', index=False, encoding='utf-8-sig')
print('Notícias analisadas:', len(sentimentos_df))
print('Modelo:', MODELO_SENTIMENTOS, '| truncamento:', MAX_TOKENS_SENTIMENTOS, 'tokens')
display(sentimentos_df[['titulo', 'sentimento', 'confianca_modelo']])


### 1.1. Distribuição e análise crítica

Leia alguns títulos e compare com o rótulo previsto. **A confiança do modelo não é uma medida de acerto.** Algumas reportagens descrevem derrotas, lesões, contratações e disputas com linguagem que pode parecer negativa/positiva, mesmo quando a notícia é informativa. Para relatar precisão, seria preciso uma avaliação humana de textos rotulados.

In [ ]:
if sentimentos_df.empty:
    print('Sem resultados de sentimento.')
else:
    distribuicao = sentimentos_df['sentimento'].value_counts().reindex(
        ['negativo', 'neutro', 'positivo'], fill_value=0)
    display(distribuicao.rename('quantidade').reset_index().rename(columns={'index':'sentimento'}))
    ax = distribuicao.plot(kind='bar', figsize=(7, 4), rot=0, title='Sentimentos previstos nas notícias')
    ax.set_ylabel('Quantidade de notícias')
    ax.set_xlabel('Polaridade prevista')
    plt.tight_layout()
    plt.savefig(SAIDA / 'distribuicao_sentimentos.png', dpi=160, bbox_inches='tight')
    plt.show()


## 2. Reconhecimento de Entidades Nomeadas (NER)

Seguindo a Aula 9, usamos **spaCy** para identificar spans de nomes e seus tipos. Antes de extrair relações, fazemos uma **inspeção de dependências sintáticas (parsing)** de uma frase do corpus — isso ajuda a entender como o sistema localiza sujeito, verbo e objeto.

**Nota:** o modelo `pt_core_news_sm` não tem necessariamente uma categoria separada para jogador ou clube; a classificação depende das etiquetas disponíveis e do contexto. Não trate toda organização reconhecida como clube.

In [ ]:
texto_exemplo = amostra.iloc[0]['_texto_unidade3'][:LIMITE_CARACTERES]
doc_exemplo = nlp(texto_exemplo)
sentenca_exemplo = next(iter(doc_exemplo.sents))
parsing_df = pd.DataFrame([
    {'token':t.text, 'lema':t.lemma_, 'classe':t.pos_,
     'dependencia':t.dep_, 'nucleo':t.head.text}
    for t in sentenca_exemplo
])
print('Título:',amostra.iloc[0]['_titulo_unidade3'])
print('Sentença:',sentenca_exemplo.text)
display(parsing_df)
displacy.render(sentenca_exemplo, style='dep', jupyter=True,
                options={'compact':True,'distance':90})

### 2.1. Identificação de entidades nas notícias

O modelo português `pt_core_news_sm` tem taxonomia própria. Verificamos seus rótulos em tempo de execução; não presumimos que ele sempre separe **jogador, clube, campeonato ou estádio**. Também conservamos URL e frase de origem para conferência.

In [ ]:
print('Categorias suportadas pelo modelo:', nlp.get_pipe('ner').labels)
linhas_ner=[]
processados=[]
for ix, doc in enumerate(nlp.pipe(amostra['_texto_unidade3'].str.slice(0,LIMITE_CARACTERES).tolist(), batch_size=8)):
    linha=amostra.iloc[ix]
    processados.append((ix,doc))
    for entidade in doc.ents:
        linhas_ner.append({
            'id_amostra':ix,'titulo':linha['_titulo_unidade3'],
            'url':linha['_url_unidade3'],
            'entidade':entidade.text,'categoria':entidade.label_,
            'inicio':entidade.start_char,'fim':entidade.end_char,
            'sentenca_origem':entidade.sent.text.strip(),
            'correta_manual':''
        })
ner_df=pd.DataFrame(linhas_ner,columns=['id_amostra','titulo','url','entidade',
                                       'categoria','inicio','fim','sentenca_origem','correta_manual'])
ner_df.to_csv(SAIDA/'entidades_ner_revisao_manual.csv',index=False)
print('Entidades reconhecidas:',len(ner_df))
display(ner_df.head(25))

In [ ]:
if not ner_df.empty:
    frequencia = (ner_df.groupby(['entidade','categoria']).size()
                  .sort_values(ascending=False).head(15))
    display(frequencia.rename('frequencia').reset_index())
    print('Distribuição das categorias:')
    display(ner_df['categoria'].value_counts().rename_axis('categoria').reset_index(name='quantidade'))
else:
    print('Nenhuma entidade detectada nesta amostra. Examine os textos e o modelo.')
displacy.render(doc_exemplo, style='ent', jupyter=True)

In [ ]:
# Exemplo visual: entidades reconhecidas na primeira notícia da amostra
if processados:
    indice_exemplo, doc_ner_exemplo = processados[0]
    print('Título:', amostra.iloc[indice_exemplo]['_titulo_unidade3'])
    displacy.render(doc_ner_exemplo[:min(120, len(doc_ner_exemplo))], style='ent', jupyter=True)


## 3. Extração de relações

Reaproveitamos o **parsing de dependências** e os **spans do NER**, tal como na Aula 9:

- Para cada verbo, buscamos um sujeito (`nsubj` / `nsubj:pass`) e objeto ou complemento (`obj`, `iobj`, `obl`).
- A relação recebe o **lema do verbo** (`contratou → contratar`). Se houver complemento oblíquo, tentamos incluir a preposição (`atuar_em`).
- Preferimos o nome da entidade detectada pelo NER quando o núcleo sintático está dentro dela.

A saída contém **relações candidatas**, que podem estar erradas; frases na voz passiva, coordenações, pronomes e dependências mal reconhecidas dificultam a extração.

In [ ]:
PREPOSICOES = {
    'no':'em','na':'em','nos':'em','nas':'em',
    'do':'de','da':'de','dos':'de','das':'de',
    'ao':'a','aos':'a','à':'a','às':'a',
    'pelo':'por','pela':'por','pelos':'por','pelas':'por'
}


def argumento(token, doc):
    entidade = next((e for e in doc.ents if e.start <= token.i < e.end), None)
    if entidade is not None:
        return entidade.text, entidade.label_
    arvore = sorted((t for t in token.subtree if not t.is_punct), key=lambda t: t.i)
    while arvore and (arvore[0].dep_ in {'det','case'} or arvore[0].pos_ in {'ADP','DET'}):
        arvore.pop(0)
    texto = ' '.join(t.text for t in arvore).strip()
    return texto[:160], ''


def extrair_relacoes_doc(doc, noticia_id, titulo='', url=''):
    linhas = []
    for sent in doc.sents:
        for verbo in sent:
            if verbo.pos_ != 'VERB':
                continue
            sujeitos = [t for t in verbo.children if t.dep_ in {'nsubj','nsubj:pass'}]
            complementos = [t for t in verbo.children if t.dep_ in {'obj','iobj','obl'}]
            for sujeito in sujeitos:
                nome_sujeito, cat_sujeito = argumento(sujeito, doc)
                for comp in complementos:
                    nome_objeto, cat_objeto = argumento(comp, doc)
                    if not nome_sujeito or not nome_objeto:
                        continue
                    prep = next((t for t in comp.children if t.dep_ == 'case'), None)
                    rel = verbo.lemma_.lower() or verbo.text.lower()
                    if comp.dep_ == 'obl' and prep is not None:
                        rel += '_' + PREPOSICOES.get(prep.text.lower(), prep.lemma_.lower())
                    linhas.append({
                        'id_amostra':noticia_id, 'titulo':titulo,'url':url,
                        'sujeito':nome_sujeito, 'categoria_sujeito':cat_sujeito,
                        'relacao':rel,'objeto':nome_objeto,'categoria_objeto':cat_objeto,
                        'sentenca_origem':sent.text.strip(),
                        'possui_ner':bool(cat_sujeito or cat_objeto),
                        'correta_manual':''
                    })
    return linhas

linhas_rel=[]
for ix, doc in processados:
    linha=amostra.iloc[ix]
    linhas_rel.extend(extrair_relacoes_doc(doc,ix,linha['_titulo_unidade3'],linha['_url_unidade3']))
colunas=['id_amostra','titulo','url','sujeito','categoria_sujeito','relacao',
         'objeto','categoria_objeto','sentenca_origem','possui_ner','correta_manual']
relacoes_df=pd.DataFrame(linhas_rel,columns=colunas)
relacoes_df.to_csv(SAIDA/'relacoes_revisao_manual.csv',index=False)
print('Triplas candidatas:',len(relacoes_df))
display(relacoes_df.head(30))

### 3.1. Avaliação manual de NER e relações

Edite os CSVs gerados e marque **1** (correto) ou **0** (incorreto) nas colunas `correta_manual`. Erros devem ser descritos no relatório: por exemplo, entidade ausente, nome incompleto, confusão entre clube e local, objeto errado ou relação sem contexto. Sem anotação humana, **não informe precisão**.

In [ ]:
print('Arquivos para revisão:')
print(SAIDA/'entidades_ner_revisao_manual.csv')
print(SAIDA/'relacoes_revisao_manual.csv')
print('Exemplo de tripla candidata e sentença de origem:')
if relacoes_df.empty:
    print('Nenhuma relação coberta pelas regras nesta amostra.')
else:
    display(relacoes_df[['sujeito','relacao','objeto','sentenca_origem']].head(10))

### 3.2. Grafo dirigido de entidades e relações

Para legibilidade, incluímos no máximo 20 triplas que tenham ao menos uma entidade identificada pelo modelo. O grafo mostra **o que as regras extraíram**, não conhecimento verificado; cada nó mantém o nome textual, sem resolução de apelidos (ex.: `Timão` e `Corinthians`).

In [ ]:
triplas_grafo = (relacoes_df[relacoes_df['possui_ner']]
                .drop_duplicates(['sujeito','relacao','objeto']).head(20))
G = nx.DiGraph()
for linha in triplas_grafo.itertuples(index=False):
    if G.has_edge(linha.sujeito, linha.objeto):
        G[linha.sujeito][linha.objeto]['relacoes'].add(linha.relacao)
    else:
        G.add_edge(linha.sujeito, linha.objeto, relacoes={linha.relacao})
if G.number_of_edges() == 0:
    print('Nenhuma relação associada a NER para construir grafo.')
else:
    fig, ax=plt.subplots(figsize=(15,10))
    pos=nx.spring_layout(G, seed=SEMENTE)
    nx.draw_networkx_nodes(G,pos,node_color='#dceaf5',node_size=2200,ax=ax)
    rotulos={n:textwrap.fill(str(n),20) for n in G.nodes()}
    nx.draw_networkx_labels(G,pos,labels=rotulos,font_size=8,ax=ax)
    nx.draw_networkx_edges(G,pos,arrows=True,arrowsize=17,ax=ax)
    edge_labels={(u,v):textwrap.fill(', '.join(sorted(d['relacoes'])),18)
                 for u,v,d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G,pos,edge_labels=edge_labels,font_size=7,ax=ax)
    ax.axis('off')
    ax.set_title('Relações candidatas extraídas das notícias do Lance!')
    plt.savefig(SAIDA/'grafo_relacoes_lance.png',dpi=170,bbox_inches='tight')
    plt.show()
    print('Nós:',G.number_of_nodes(),'| Arestas:',G.number_of_edges())

## 4. Resumo e interpretação para a entrega de hoje

Use as tabelas e imagens salvas em `resultados/` para explicar:

1. **Sentimentos:** quantas notícias foram avaliadas, distribuição dos rótulos e ao menos um exemplo discutido criticamente.
2. **NER:** quantas entidades foram reconhecidas, quais categorias apareceram e exemplos de acertos/erros.
3. **Relações:** quantas triplas candidatas foram extraídas, exemplos relevantes, erros e o grafo.

**Atenção:** *sem revisão humana, não declare acurácia/precisão*. Evite afirmar que notícias, sentimentos e relações gerados automaticamente são necessariamente verdadeiros.

O PDF do Trabalho da Unidade 3 permite amostras menores **desde que explicitadas** e exige mencionar ferramentas de IA utilizadas, incluindo fornecedor, modelo e prompts. Neste notebook, cite também o modelo de sentimento e o spaCy usados no experimento.

In [ ]:
resumo = {
    'base': str(caminho_base),
    'noticias_disponiveis': len(base),
    'amostra_ner_relacoes': len(amostra),
    'amostra_sentimentos': len(sentimentos_df),
    'limite_caracteres_ner_relacoes': LIMITE_CARACTERES,
    'limite_tokens_sentimentos': MAX_TOKENS_SENTIMENTOS,
    'modelo_sentimentos': MODELO_SENTIMENTOS,
    'modelo_ner_relacoes': 'spacy/pt_core_news_sm',
    'entidades_reconhecidas': len(ner_df),
    'triplas_candidatas': len(relacoes_df),
    'distribuicao_sentimentos': sentimentos_df['sentimento'].value_counts().to_dict(),
}
with (SAIDA / 'resumo_experimentos.json').open('w', encoding='utf-8') as arquivo:
    json.dump(resumo, arquivo, ensure_ascii=False, indent=2)
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print()
print('Arquivos produzidos em:', SAIDA.resolve())
print('- sentimentos_lance.csv / distribuicao_sentimentos.png')
print('- entidades_ner_revisao_manual.csv / relacoes_revisao_manual.csv')
print('- grafo_relacoes_lance.png (se houver relações para desenhar)')
print('- resumo_experimentos.json')
